# P1｜AST Joint-Native Reference

**状态：Preflight / Adapter HOLD；首轮 core。** 目的：四个数据集都通过同一个 frozen AST candidate、同一个 source-time sliding-window interface 与同一个 trainable shared projector；只保留 dataset-native aggregation、heads、loss 和 metrics。

## 唯一变量、匹配对照与四数据集边界

P1 是 AST shared-window reference，不引入 eligibility、balanced sampler、PAFA、MVST、LODO。四数据集 native unit 先按 Proposed Benchmark Policy 切为 2.0 s window / 1.0 s stride；短 unit 只 zero-pad，长 unit补一个不重复的 end-aligned tail window。所有 window 经过同一个 frozen AST，再经过同一个 trainable Linear 768→256 projector。AST frontend 对每个 2 s source window 只能内部 zero-pad 到已审计 8 s grid，不得 repeat 或截断；因此是 AST+adapter package-level reference。

P2–P5 必须匹配 window policy、四数据集 split/group、shared projector、native heads、source-proportional sampler、budget、seed 与 validation-only selection；候选 package 的 encoder/frontend/dimension adapter 是唯一允许变化。该任务不是统一标签学习。

- ICBHI：cycle → flat4 [B,4]；SPRSound：event → binary [B,2] 与 raw7 [B,7]；KAUH：recording → raw9 [B,9]，B/D/E 同 patient group。
- HF：15 秒 recording → window embeddings [B,K,768] → shared projected sequence [B,K,256] → native I/E/CAS/DAS logits [B,K,4]；四通道 mask 独立，gap/missing/unknown 不是 raw negative。paper-native rasterization 的零仅是 source-task constructed negative。
- ICBHI/SPRSound/KAUH 对 valid window embeddings 做 masked mean，再进入各自 native heads。指标按 dataset/native task 分开，禁止跨数据集 pooled Score。

In [ ]:
from pathlib import Path
import os
from baseline.multidataset_pipeline.preflight import P1_P5_SELECTION_RULE, P1_P5_UPDATE_BUDGET, freeze_receipt

PIPELINE = {
    "id": "P1",
    "seed": 20260728,
    "split_policy": "immutable_prerequisite_receipts",
    "encoder": "AST_AudioSet_shared_across_four_datasets",
    "encoder_scope": "frozen_pretrained_AST",
    "shared_projector": "minimal_linear_projector_768_to_256",
    "shared_projector_bias": True,
    "shared_projector_lanes": ["ICBHI", "SPRSound", "HF", "KAUH"],
    "window_policy": "source_time_2s_window_1s_stride",
    "sampler": "four_dataset_source_proportional",
    "hf_lane": "shared_projected_window_sequence_to_temporal4_head",
    "hf_uses_shared_projector": True,
    "trainable_scope": "one_shared_projector_768_to_256_plus_four_dataset_native_heads",
    "contract_modules": ["baseline.multidataset_pipeline.contracts", "baseline.multidataset_pipeline.sliding_window", "baseline.multidataset_pipeline.joint_native", "baseline.multidataset_pipeline.preflight"],
    "engineering_test": "tests/test_multidataset_pipeline.py::JointNativeContractTest",
    "batch_size": 8,
    "update_budget": P1_P5_UPDATE_BUDGET,
    "validation_interval_updates": 1725,
    "selection": P1_P5_SELECTION_RULE,
    "output_dir": "result/reproduce/P1_joint_native_reference",
    "receipt_path": "result/reproduce/P1_joint_native_reference/P1_receipt.json",
}
PROJECT_ROOT = Path(os.environ.get("ACOUSTIC_PROJECT_ROOT", Path.cwd())).resolve()
APPROVAL_RECEIPT = os.environ.get("P1_APPROVAL_RECEIPT")


## 科学 gate 与运行前审批

seed=20260728、batch size=8、四数据集 subtrain units=13794、1725 updates/reference epoch、总 budget=86250 与 validation-only selection 已冻结；selection scalar 只选 checkpoint，不是 pooled performance。仍须闭合 AST shared-window production adapter、checkpoint/source SHA、2 s→8 s internal zero-pad invariance、四 dataset lineage/mask、HF target counts、CUDA smoke、approval 与新 independent verifier。outer/test 不得用于 window、selection 或执行前决策。

In [ ]:
PREFLIGHT = freeze_receipt()
required = [PIPELINE["update_budget"], PIPELINE["selection"], APPROVAL_RECEIPT]
if any(value in (None, "") for value in required):
    raise RuntimeError("P1 fail closed: approval receipt is missing")
approval_path = PROJECT_ROOT / APPROVAL_RECEIPT
if not approval_path.is_file():
    raise FileNotFoundError(approval_path)
DRY_RUN_PLAN = {"pipeline": PIPELINE, "approval": str(approval_path), "execute": False}


## 输出、receipt 与结果表

receipt 至少记录 pipeline/config/manifest/checkpoint hashes、2 s/1 s window contract、per-dataset window/valid counts、four-lane projector gradient scope、HF constructed-negative semantics、split/group hashes、seed/updates、selection、native metrics、warnings 与 verifier status。

| Dataset / native task | Metric | Result | Decision |
|---|---:|---:|---|
| ICBHI / SPRSound / HF / KAUH | 待冻结 | Not run | 未判定 |

**Test Result = Not run。Decision = HOLD。Claim boundary：若执行，只能称四数据集 shared-window AST+adapter package reference；不是统一标签任务，也不是性能结果。**